<a href="https://colab.research.google.com/github/josenomberto/UTEC-CDIAV3-MCD8014/blob/main/practica_dirigida_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica Dirigida 01: KNN en CPU (OpenMP) y GPU (CUDA)
**Curso**: Applied High Performance Computing  
**Institución**: UTEC  
**Profesor**: Applied HPC Team  
**Estudiante**:
- Herles Alejandro Pinedo
- David Jimenez
- José Carlos Nomberto



## PREGUNTA 1

### Formula Matemática:

Considerando 3 operaciones por Q, la formula para el calculo de FLOP es:
$$\text{FLOPs} = 3 * Q * N * d$$



### Cálculos para el menor y mayor $Q$ ($N = 100,000$, $d = 64$):

#### A. Menor consulta ($Q = 1,000$):
$$T_s(1000) = 3 \cdot 1,000 \cdot 100,000 \cdot 64 = 1.92 \times 10^{10} \text{ FLOPs} = 19.2 \text{ GFLOPs}$$

#### B. Mayor consulta ($Q = 1,000,000$):
$$T_s(1000000) = 3 \cdot 1,000,000 \cdot 100,000 \cdot 64 = 1.92 \times 10^{13} \text{ FLOPs} = 19.2 \text{ TFLOPs}$$

### Comparación:
El meayor caso tiene 1000 más trabajo que el menor caso ya que el #FLOPs crece linealmente con Q al ser N y d constantes



## PREGUNTA 2

Considerando la GPU NVIDIA A100 de Khipu con:
- $SM = 108$ multiprocesadores
- Límite de hardware de hilos residentes máximos por SM = $2048$
- Hilos residentes máximos teóricos del dispositivo = $108 \cdot 2048 = 221,184$ hilos.

Utilizamos las fórmulas para calcular la ocupación teórica reportada por CUDA para cada tamaño de bloque ($TPB \in \{128, 256, 512, 1024\}$):
$$\text{residentBlocks} = SM \cdot \text{blocksPerSM}$$
$$\text{residentThreads} = \text{residentBlocks} \cdot TPB$$

### Tabla de Resultados:

| $TPB$ | $\text{blocksPerSM}$ | $\text{residentBlocks}$ | $\text{residentThreads}$ |
| :---: | :---: | :---: | :---: |
| **128** | 16 | $108 \times 16 = 1728$ | $1728 \times 128 = 221,184$
| **256** | 8 | $108 \times 8 = 864$ | $864 \times 256 = 221,184$
| **512** | 4 | $108 \times 4 = 432$ | $432 \times 512 = 221,184$
| **1024** | 2 | $108 \times 2 = 216$ | $216 \times 1024 = 221,184$

### Interpretación:
1. Observamos que en los cuatro TPB, el total de hilos residentes es exactamente $221,184$, el que coincide con el límite físico de la GPU, por lo que  todas las configuraciones logran una ocupación máxima.
2. Con **TPB = 128**, la cantidad de bloques es grande ($1728$) lo que genera mejor granularidad para balancear la carga de trabajo entre SMs, pero puede aumentar el costo de planificación
Con **TPB = 1024** se reducen los bloques a $216$, lo que minimiza la sobrecarga de planificación, pero reduce la eficiencia.




## PREGUNTA 3

Para evaluar la eficiencia del kernel CUDA con $TPB = 256$ y $\text{residentBlocks} = 864$, calculamos:
1. $\text{blocksPerGrid} = \lceil Q / TPB \rceil$
2. $\text{waves} = \lceil \text{blocksPerGrid} / \text{residentBlocks} \rceil$

### Tabla de Oleadas:

| $Q$ | $\text{blocksPerGrid}$ | $\text{waves} $ | # waves | Estado de la GPU |
| :---: | :---: | :---: | :---: | :---: |
| **1,000** | 4 | $4 / 864 \approx 0.0046$ | 1 | Subutilizada (Extrema) |
| **10,000** | 40 | $40 / 864 \approx 0.0463$ | 1 | Subutilizada (Alta) |
| **50,000** | 196 | $196 / 864 \approx 0.2269$ | 1 | Subutilizada (Moderada) |
| **100,000** | 391 | $391 / 864 \approx 0.4525$ | 1 | Subutilizada (Leve) |
| **200,000** | 782 | $782 / 864 \approx 0.9051$ | 1 | Subutilizada(Cercana a 1 ola) |
| **500,000** | 1,954 | $1,954 / 864 \approx 2.2616$ | 3 | Trabajando por oleadas (con efecto cola) |
| **1,000,000** | 3,907 | $3,907 / 864 \approx 4.5220$ | 5 | Trabajando por oleadas (Saturación asintótica) |



### Explicación de los Estados de la GPU:

Observamos que que con $Q = 200,000$ (782 bloques), se aprovecha el $90.5\%$ de los slots disponibles de la GPU en una sola oleada rápida. Valores por debajo generan subutilización grande y por encima generan saturación con oleadas



## PREGUNTA 4

### Tiempo de GPU :  $T_{\text{GPU,total}} = T_{\text{H2D}} + T_{\text{kernel}} + T_{\text{D2H}}$
El tiempo de GPU se debe separar ya que el CPU y GPU usan memorias físicas separadas y existe comunicación entre ellas para copiar los datos entre ellas:
1. $T_{\text{H2D}}$ (Host-to-Device): Tiempo de copia de los datos desde la RAM del CPU a la memoria global de la GPU .
2. $T_{\text{kernel}}$: Tiempo de cómputo interno de los multiprocesadores ejecutando la lógica del KNN.
3. $T_{\text{D2H}}$ (Device-to-Host): Tiempo de copia de los resultados desde GPU al CPU.
Al separar los tiempos se puede identificar si una demora es en la transferencia de datos o en el computo



### Intensidad Aritmética ($AI$)
La Intensidad Aritmética se define como:
$$AI = \frac{\#\text{FLOPs}}{\text{Bytes transferidos}}$$

Bajo la consideración del problema, por cada comparación individual (par query-train de dimensión $d$):
- Hacemos $3d$ FLOPs ($d$ restas, $d$ multiplicaciones, $d$ sumas).
- Transferimos (leemos/escribimos) $8d$ Bytes.

Por lo tanto, la intensidad aritmética teórica del kernel KNN es constante respecto a $d$:
$$AI = \frac{3d \text{ FLOP}}{8d \text{ Byte}} = 0.375 \text{ FLOP/Byte}$$


## Memory-Bound de KNN

Dado que la intensidad aritmética de KNN  es un valor bajo ($0.375 \text{ FLOP/Byte}$), entonces tenemos pocas operaciones por byte, por lo que KNN es de naturaleza Memory-bound.


